# Mutation, Variation, and the Sources of Novelty Workflow

This notebook scaffold supports the article **Mutation, Variation, and the Sources of Novelty**. It can be expanded with mutation supply, mutation spectra, sequence distance, nucleotide diversity, structural variation, novelty condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
spectrum = pd.read_csv(article_dir / 'data' / 'mutation_spectrum.csv')
spectrum['fraction'] = spectrum['count'] / spectrum['count'].sum()
spectrum.sort_values('fraction', ascending=False).round(4)

In [ ]:
mu = 1e-8
target_length = 1.2e8
n_genomes = 500
lambda_expected = n_genomes * target_length * mu
pd.DataFrame({'expected_mutations_lambda':[lambda_expected]}).round(6)

In [ ]:
geno = pd.read_csv(article_dir / 'data' / 'genotype_site_summary.csv')
geno['p'] = geno['derived_count'] / geno['n_chromosomes']
geno['pi_site'] = 2 * geno['p'] * (1 - geno['p'])
geno['segregating'] = (geno['p'] > 0) & (geno['p'] < 1)
pd.DataFrame({'n_sites':[len(geno)], 'segregating_sites':[geno['segregating'].sum()], 'pi':[geno['pi_site'].mean()]}).round(5)

In [ ]:
sv = pd.read_csv(article_dir / 'data' / 'structural_variants.csv')
sv['rarity_score'] = 1 - sv['population_frequency']
sv['functional_priority_score'] = (
    0.35 * sv['overlaps_gene'].astype(float) +
    0.30 * sv['overlaps_regulatory_region'].astype(float) +
    0.20 * sv['rarity_score'] +
    0.15 * (sv['size_bp'] / sv['size_bp'].max())
)
sv.sort_values('functional_priority_score', ascending=False).round(4)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'novelty_condition_sites.csv')
condition['novelty_condition_score'] = (
    0.15 * condition['mutation_supply'] +
    0.17 * condition['standing_variation'] +
    0.14 * condition['recombination_potential'] +
    0.15 * condition['regulatory_flexibility'] +
    0.15 * condition['developmental_modularity'] +
    0.14 * condition['ecological_opportunity'] +
    0.10 * (1 - condition['constraint_risk'])
)
condition.sort_values('novelty_condition_score', ascending=False).round(3)